### Step 1: Creating Tokens

In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text=f.read()


print("total number of characters: ", len(raw_text))
print(raw_text[:99])

total number of characters:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
import re #Regular expression
text="Hello, world. This , is a test."
result=re.split(r'(\s)',text) #it slpits the data on the basis of white spaces. (\s) denotes the white space.
print(result)

['Hello,', ' ', 'world.', ' ', 'This', ' ', ',', ' ', 'is', ' ', 'a', ' ', 'test.']


In [3]:
# Now we also want to split the on the basis of white spaces,commas,fullstops,etc.
result=re.split(r'([,.]|\s)',text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ' ', '', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [4]:
# there is still issue with this kind of spliting such as list still includes the whitespace characters.
# so for removing that redundant charecters safely as follows:
result=[item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [5]:
# now we create the saperate token of special charecters
text="Hello, world. Is this-- a test?"
result = re.split(r'([.,:?_!"()\']|--|\s)',text)
result =[item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [6]:
preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed= [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
4690


### Step 2: Creating Token IDs

In [7]:
# now we short them alphabetically to determine the vocabulary size: 
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)

print(vocab_size)

1130


In [8]:
vocab ={token:integer for integer,token in enumerate(all_words)}

In [9]:
#encoder
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 10:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


In [10]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed=re.split(r'([,.?:;_!"()\']|--|\s)',text)
        preprocessed=[item.strip()for item in preprocessed if item.strip()]
        ids=[self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        text=" ".join([self.int_to_str[i]for i in ids])
        # replacing the spaces before the specified punctuations
        text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [11]:
tokenizer=SimpleTokenizerV1(vocab)

text=""""It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [12]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

#### Adding special context tokens

In the previous section, we implemented a simple tokenizer and applied it to a passage from the training set.

so we need to modify the tokenizer to handle the unknown words.

In particular, we will modify the vocabulary and tokenizer we implemented in the previous section, SimpleTokenizerV2 to support two tokens,<|unk|> and <|endoftext|>

In [13]:
all_tokens=sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])

vocab={token:integer for integer, token in enumerate(all_tokens)}

In [14]:
len(vocab.items())

1132

In [15]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)
